# USGS dataretrieval Python Package Statistics Examples

This notebook provides examples of using the Python dataretrieval package to retrieve summary statistics for observed variables at a United States Geological Survey (USGS) monitoring location using the **USGS Water Data API** via the `waterdata` module. The `waterdata` module is the recommended way to access USGS water data and replaces the deprecated `nwis` module.

Two statistics functions are demonstrated:

* `get_stats_date_range()` — monthly, calendar-year, and water-year summaries (the "observationIntervals" service).
* `get_stats_por()` — day-of-year and month-of-year summaries over the full period of record (the "observationNormals" service).

### Install the Package

Use the following code to install the package if it doesn't exist already within your Jupyter Python environment.

In [ ]:
!pip install dataretrieval

Load the package so you can use it along with other packages used in this notebook.

In [ ]:
from IPython.display import display
from matplotlib import ticker

from dataretrieval import waterdata


### Basic Usage

This example uses `get_stats_date_range()` to retrieve monthly and annual statistics for an observed variable at a USGS monitoring location. Commonly used arguments include:

* **monitoring_location_id** (string or list of strings): USGS monitoring location id(s), formed as the agency code and site number joined by a hyphen (e.g. `"USGS-02319394"`).
* **parameter_code** (string or list of strings): 5-digit USGS parameter code(s), e.g. `"00060"` (discharge).
* **computation_type** (string or list of strings): the statistic(s) to compute — one or more of `arithmetic_mean`, `maximum`, `median`, `minimum`, `percentile`.
* **start_date** / **end_date** (string): optionally bound the period summarized, in `YYYY-MM-DD` format.

#### Example 1: Get monthly and annual mean discharge for a single monitoring location

In [ ]:
# Set the parameters needed to retrieve data
site = "USGS-02319394"
parameter_code = "00060"  # Discharge

# Retrieve the statistics (monthly, calendar-year, and water-year means)
x1 = waterdata.get_stats_date_range(
    monitoring_location_id=site,
    parameter_code=parameter_code,
    computation_type="arithmetic_mean",
)
print("Retrieved " + str(len(x1[0])) + " statistic values.")

### Interpreting the Result

Each `waterdata` function returns a tuple of a pandas data frame and a metadata object. The data frame holds the computed statistics; each row is one interval, identified by the `interval_type` column (`month`, `calendar_year`, or `water_year`), with the computed statistic in the `value` column.

Once you've got the data frame, there are several useful things you can do to explore the data.

In [ ]:
# Display the data frame as a table
display(x1[0])

Show the data types of the columns in the resulting data frame.

In [ ]:
print(x1[0].dtypes)

Make a quick time series plot of the annual (calendar-year) mean values.

In [ ]:
# select the annual (calendar-year) means into a plain DataFrame for plotting.
# The statistics services return a GeoDataFrame carrying a site-point geometry,
# and report numeric values as strings, so we coerce ``value`` to float.
annual = x1[0].loc[
    x1[0]["interval_type"] == "calendar_year", ["start_date", "value"]
].copy()
annual["year"] = annual["start_date"].str[:4].astype(int)
annual["value"] = annual["value"].astype(float)
annual = annual.sort_values("year")

ax = annual.plot(x="year", y="value", legend=False)
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%d"))
ax.set_xlabel("Year")
ax.set_ylabel("Annual mean discharge (cfs)")

The other part of the result is a metadata object describing the query that was executed. For example, you can access the URL that was assembled to retrieve the requested data from the USGS Water Data API.

In [ ]:
print("The query URL used to retrieve the data was: " + x1[1].url)

### Additional Examples

#### Example 2: Get monthly and annual mean statistics for two monitoring locations

Multiple monitoring locations and parameter codes can be requested at once; only the data that are available are returned.

In [ ]:
x2 = waterdata.get_stats_date_range(
    monitoring_location_id=["USGS-02319394", "USGS-02171500"],
    parameter_code=["00010", "00060"],
    computation_type="arithmetic_mean",
)
display(x2[0])

#### Example 3: Day-of-year mean and median statistics over the period of record

`get_stats_por()` summarizes the full period of record by day of year (and month of year). Here we request both the mean and median daily statistics for discharge at a monitoring location.

In [ ]:
x3 = waterdata.get_stats_por(
    monitoring_location_id="USGS-02171500",
    parameter_code="00060",
    computation_type=["arithmetic_mean", "median"],
)
display(x3[0])